In [6]:
# Kendi kurduğumuz feature extraction pipeline'nın çıktısı ile modelin checkpoint'in beklediği girdi arasıda uyumsuzluk var
# bizim visiual channel -> [N, 2048] üretiyor (Inception GAP)
# checkpoint'in visual_proj ise 1024-d bekliyor.
# önce bu uyumsuzluğun sebebini bulup daha sonra çözüm buluruz.


# checkpoint -> belirli bir adımdaki ağırlıklar, kaydedilmiş hali. sadece bir sayıdır, modelin kendisini vermez.
# önce model kurulur, mimari hazırlanır daha sonra bu checkpoint yüklenerek mimari çalıştırılır.
# böylece model artık eğitilmiş olur.
# checkpoint'i yüklerken model ve checkpoint değerleri eşleşmelidir.
# hem isim olarak hem de shape olarak - buradaki sorunumuz bu.


In [7]:
! pip install -q huggingface_hub

In [8]:
# Yazarların Mr. HiSum eğitimi sırasında validation'da en iyi sonucu aldıkları anda çektikleri bir checkpoint noktası

from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id = "smkim37/TripleSumm",
    filename="best_model_ckpt_mrhisum.pth",
    # repo_type="dataset"
)

file_path

'/root/.cache/huggingface/hub/models--smkim37--TripleSumm/snapshots/fbfd318abc4d317a2618a8f9b03bf138a30662c2/best_model_ckpt_mrhisum.pth'

In [9]:
import torch

print(type(torch.load(file_path, map_location="cpu", weights_only=True)))

sd = torch.load(file_path, map_location="cpu", weights_only=True) # indirdiğimiz checkpoint'i yükleyelim.

for i, k in sd.items():
  print(i, " shape: ", k.shape)

# <class 'collections.OrderedDict'>
# state_dict, PyTorch'ta bir modelin öğrenilmiş parametrelerini (weights ve bias'ları) ve bazı katmanların durum bilgilerini tutan bir Python sözlüğüdür (dict).
# yani aslında modelin o ana kadar öğrendiği ağırlıkların bir listesi
# state_dict yalnızca öğrenilebilir, yani eğitim boyunca güncellenen değerleri ve buffer'ları tutar.

# model koduna baktığımızda:

#        self.head = nn.Sequential(
#            nn.Linear(input_dim, hidden_dim),
#            nn.GELU(),
#            nn.Dropout(dropout),
#            nn.LayerNorm(hidden_dim),
#            nn.Linear(hidden_dim, 1),
#            nn.Sigmoid()
#        )

# state_dict son elemanlarına baktığmızda ise:

# head.0.weight  shape:  torch.Size([192, 128])
# head.0.bias  shape:  torch.Size([192])
# head.3.weight  shape:  torch.Size([192])
# head.3.bias  shape:  torch.Size([192])
# head.4.weight  shape:  torch.Size([1, 192])
# head.4.bias  shape:  torch.Size([1])

# olduğunu görürüz. head.1, head.2 ve head.5 state_dict'e kaydedilmemiş.
# sebebi onların birer matematiksel işlem ve öğrenilebilir bir parametre olmadığı için

<class 'collections.OrderedDict'>
visual_proj.weight  shape:  torch.Size([128, 1024])
visual_proj.bias  shape:  torch.Size([128])
text_proj.weight  shape:  torch.Size([128, 768])
text_proj.bias  shape:  torch.Size([128])
audio_proj.weight  shape:  torch.Size([128, 768])
audio_proj.bias  shape:  torch.Size([128])
visual_ln.weight  shape:  torch.Size([128])
visual_ln.bias  shape:  torch.Size([128])
text_ln.weight  shape:  torch.Size([128])
text_ln.bias  shape:  torch.Size([128])
audio_ln.weight  shape:  torch.Size([128])
audio_ln.bias  shape:  torch.Size([128])
temporal_pe.pe  shape:  torch.Size([1, 10000, 128])
modality_embedding.embedding.weight  shape:  torch.Size([4, 128])
temporal_block.0.wsa_layer.wsa.qkv_proj.weight  shape:  torch.Size([384, 128])
temporal_block.0.wsa_layer.wsa.out_proj.weight  shape:  torch.Size([128, 128])
temporal_block.0.wsa_layer.ffn.w1.weight  shape:  torch.Size([170, 128])
temporal_block.0.wsa_layer.ffn.w2.weight  shape:  torch.Size([170, 128])
temporal_blo

In [30]:
# Pooling = çok sayıda vektörü tek bir vektöre indirme işlemi.
# Problem şu: transformer tabanlı encoder'lar tek bir çıktı vermez, dizi verir
# Ama bizim pipeline "bir birim = bir vektör" istiyor. Aradaki köprü pooling.
# örneğin audio channelda yaptık:
# 10 saniyelik bir ses dilimini AST'ye verdiğimizde çıktı tek vektör yerine, spektrogramın her patch için bir vektör verdi
# yani, [patch_sayısı, 768] şeklinde bir dizi idi. Fakat biz bu dilimin tamamını temsil eden tek bir 768'lik vektör istiyorduk
# bunu çözmek için [1214, 768] → ortalama → [768] yapmıştık. İşte bu mean pooling.
# yine metin tarafında da aynısını yaptık, paddinglerden dolayı attention_mask v.s. olusturduk. bu da masked mean poolingtir.

# şimdi yazarlar kendi checkpoint'lerini nasıl pool'ladılar ? (cls, mean, max ?? )
# çünkü farklı farklı poolingler farklı sonuçlar doğurur. bunlarında eşleşmesi lazım.
# o yüzden onu bulmamız gerekiyor.


# hugging face'de modelin eğitildiği dataset repo'sunun dosyalarını uzaktan listeleyelim.
# 40 GB olduğu için indirmek yerine sadece bu klasörde neler var ona bakacağız.

from huggingface_hub import HfFileSystem # os.listdir gibi calısır
import h5py

fs = HfFileSystem() # class'tan nesneyi olustur.

fs_list = fs.ls("datasets/hminjeong/TripleSumm-Mr.HiSum", detail=False)
fs_list

['datasets/hminjeong/TripleSumm-Mr.HiSum/.gitattributes',
 'datasets/hminjeong/TripleSumm-Mr.HiSum/README.md',
 'datasets/hminjeong/TripleSumm-Mr.HiSum/mrhisum_feat_audio_ast.h5',
 'datasets/hminjeong/TripleSumm-Mr.HiSum/mrhisum_feat_text_roberta.h5',
 'datasets/hminjeong/TripleSumm-Mr.HiSum/mrhisum_feat_visual_inceptionv3.h5',
 'datasets/hminjeong/TripleSumm-Mr.HiSum/mrhisum_gt.h5',
 'datasets/hminjeong/TripleSumm-Mr.HiSum/mrhisum_metadata.csv',
 'datasets/hminjeong/TripleSumm-Mr.HiSum/mrhisum_split.json']

In [41]:
# metadata dosyası

from huggingface_hub import hf_hub_download
import h5py
import pandas as pd

file_path = hf_hub_download(
    repo_id = "hminjeong/TripleSumm-Mr.HiSum",
    filename="mrhisum_metadata.csv",
    repo_type="dataset"
)

metadata_df = pd.read_csv(file_path)

metadata_df.head(5)


,video_id,youtube_id,duration,views,labels
0,ORaA,JhdjUam0l6A,258,84554,[8]
1,IwaA,wq7rSbQx2G8,137,170768,[ 11 20 22 29 32 2604]
2,SLaA,DQntand_VXo,138,63938,[21 23 24]
3,FoaA,0DKYJvpg9l0,188,55751,[21]
4,agaA,Fi2eWcQ5zB0,181,58783,[ 0 12]


In [46]:
with fs.open("datasets/hminjeong/TripleSumm-Mr.HiSum/mrhisum_feat_audio_ast.h5", "rb", block_size=4*1024*1024, cache_type="blockcache") as f:
  h5file = h5py.File(f, "r")
  ds = h5file["ORaA"]
  print("type: ", type(ds))
  print("shape: ", ds.shape)
  print("dtype: ", ds.dtype)



type:  <class 'h5py._hl.dataset.Dataset'>
shape:  (258, 768)
dtype:  float32


In [48]:
# yukarıda
    # şimdi yazarlar kendi checkpoint'lerini nasıl pool'ladılar ? (cls, mean, max ?? )
    # çünkü farklı farklı poolingler farklı sonuçlar doğurur. bunlarında eşleşmesi lazım.
    # o yüzden onu bulmamız gerekiyor.
# demiştik. aslında paper'ı okusak buluruz bunu, bu kadar uğraşmaya gerek yok :D

# 5.1 bölümüne bakarsak extraction pipeline'ı (MoSu için):
# Frame'ler 1 fps'de örneklenip pretrained CLIP ile encode ediliyor.
# Bizim fps=1 kararımız birebir aynı, encoder farklı.)

# Metin — asıl bulgu burada:
# Pooling yöntemi:
# CLS. Bizim seçimimiz masked mean idi — yöntem farkı teyit edildi, varsayımımız onların pipeline'ıyla uyuşmuyor.
# biz mean kullanmıstık onlar CLS. değiştirecek miyiz ? - hayır.
# literatürde mean > cls deniyor. onlar neden cls kullanmıs soru işareti.
# aslında bu tez açısından da bir ablation "BERTurk mean vs CLS pooling'in TRT özetleme performansına etkisi."


In [49]:
with fs.open("datasets/hminjeong/TripleSumm-Mr.HiSum/mrhisum_feat_text_roberta.h5", "rb", block_size=4*1024*1024, cache_type="blockcache") as f:
  h5file = h5py.File(f, "r")
  ds = h5file["ORaA"]
  print("type: ", type(ds))
  print("shape: ", ds.shape)
  print("dtype: ", ds.dtype)

type:  <class 'h5py._hl.dataset.Dataset'>
shape:  (258, 768)
dtype:  float32
